# Glider Flight Simulator

**Project:** compare how different airfoils affect the glide distance of a small hand-launched glider.
**Author:** *Maja Roberts*  |  **Started:** *07/11/2026*

**Plan**
1. Model the glider as a point mass in 2D (lift, drag, weight).
2. Solve the equations numerically (Euler, then Runge-Kutta) and compare.
3. Replace the placeholder lift/drag model with real airfoil polars.
4. Predict the glide distance for each airfoil, then test the real glider and compare.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

%matplotlib inline

## 2. Constants and glider properties

Replace the placeholder values with measurements from your Fusion 360 model and your real glider.
Keep everything in SI units (metres, kilograms, seconds).

In [ ]:
g   = 9.81      # gravitational acceleration (m/s^2)
rho = 1.225     # air density at sea level (kg/m^3)

m   = 0.050     # glider mass (kg)            <- measure this
S   = 0.020     # wing area (m^2)             <- from your CAD model

# Launch conditions
h0     = 1.5    # launch height (m)
v0     = 5.0    # launch speed (m/s)
gamma0 = 0.0    # launch flight-path angle (rad), 0 = horizontal

## 3. Lift and drag model

**Placeholder model for now.** Lift coefficient rises linearly with angle of attack, and drag follows a simple parabolic polar:

- C_L = C_L0 + a * alpha
- C_D = C_D0 + k * C_L^2

Later, swap this function for interpolation of real polar data from Airfoil Tools (see section 8).

In [ ]:
def lift_drag_coeffs(alpha):
    """Return (CL, CD) for angle of attack alpha (radians). PLACEHOLDER MODEL."""
    CL0 = 0.2
    a   = 2 * np.pi * 0.8      # lift-curve slope (per radian), reduced from the ideal 2*pi
    CD0 = 0.03
    k   = 0.06
    CL = CL0 + a * alpha
    CD = CD0 + k * CL**2
    return CL, CD

alpha_trim = np.radians(4)     # fixed angle of attack the glider flies at

## 4. Equations of motion

State: position (x, y), speed v, and flight-path angle gamma.

- dx/dt = v cos(gamma)
- dy/dt = v sin(gamma)
- dv/dt = (-D - m g sin(gamma)) / m
- dgamma/dt = (L - m g cos(gamma)) / (m v)

where L = 0.5 rho v^2 S C_L and D = 0.5 rho v^2 S C_D.

**Prediction before running:** *what do you expect the trajectory to look like, and why?*

In [ ]:
def derivatives(t, state, alpha=alpha_trim):
    x, y, v, gamma = state
    CL, CD = lift_drag_coeffs(alpha)
    q = 0.5 * rho * v**2 * S          # dynamic pressure x wing area
    L, D = q * CL, q * CD
    dx     = v * np.cos(gamma)
    dy     = v * np.sin(gamma)
    dv     = (-D - m * g * np.sin(gamma)) / m
    dgamma = (L - m * g * np.cos(gamma)) / (m * v)
    return [dx, dy, dv, dgamma]

## 5. Solve with Runge-Kutta (SciPy)

An event stops the simulation when the glider reaches the ground (y = 0).

In [ ]:
def hit_ground(t, state, alpha=alpha_trim):
    return state[1]
hit_ground.terminal  = True
hit_ground.direction = -1

state0 = [0.0, h0, v0, gamma0]

sol = solve_ivp(derivatives, (0, 30), state0, events=hit_ground,
                max_step=0.01, rtol=1e-8, atol=1e-10)

x, y, v, gamma = sol.y
print(f"Glide distance: {x[-1]:.2f} m")
print(f"Flight time:    {sol.t[-1]:.2f} s")
print(f"Overall glide ratio (distance / launch height): {x[-1]/h0:.1f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x, y)
plt.xlabel("Horizontal distance (m)")
plt.ylabel("Height (m)")
plt.title("Glider trajectory (Runge-Kutta)")
plt.grid(True)
plt.show()

**Result and notes:** *what did you get? Does it match your prediction? What surprised you?*

## 6. Compare with Euler's method

Euler is the simplest numerical method, so it makes a good baseline. Compare it to Runge-Kutta for different time steps to see how numerical error behaves.

In [ ]:
def euler_flight(dt, alpha=alpha_trim, t_max=30):
    state = np.array(state0, dtype=float)
    xs, ys = [state[0]], [state[1]]
    t = 0.0
    while t < t_max and state[1] > 0:
        state = state + dt * np.array(derivatives(t, state, alpha))
        t += dt
        xs.append(state[0]); ys.append(state[1])
    return np.array(xs), np.array(ys)

for dt in [0.1, 0.01, 0.001]:
    xe, ye = euler_flight(dt)
    print(f"dt = {dt:<6} Euler glide distance = {xe[-1]:.2f} m   (RK reference: {x[-1]:.2f} m)")

**Notes:** *how does the Euler result change as dt shrinks? Which step size is good enough, and why?*

## 7. Sanity checks

Before trusting the model, check it behaves sensibly.

- Does the best glide ratio occur at the angle of attack where C_L / C_D is highest?
- Does the glide distance go up with launch height and launch speed in the way you'd expect?
- Do the numbers stay physical (no negative speeds, no crazy loops)?

In [ ]:
alphas = np.radians(np.linspace(-2, 12, 100))
CL, CD = lift_drag_coeffs(alphas)

plt.figure(figsize=(8, 4))
plt.plot(np.degrees(alphas), CL / CD)
plt.xlabel("Angle of attack (degrees)")
plt.ylabel("Lift-to-drag ratio, CL/CD")
plt.title("L/D versus angle of attack")
plt.grid(True)
plt.show()

best = np.argmax(CL / CD)
print(f"Best L/D = {(CL/CD)[best]:.1f} at alpha = {np.degrees(alphas[best]):.1f} degrees")

## 8. Swap in real airfoil data

1. Download polars for each airfoil from **Airfoil Tools** (airfoiltools.com) at a suitable Reynolds number, saved as CSV.
2. Load them with `np.loadtxt` or `pandas.read_csv`.
3. Replace `lift_drag_coeffs` with an interpolating version, for example:

```python
# Example structure - adjust column indices to match your file
data = np.loadtxt("naca2412_polar.csv", delimiter=",", skiprows=11)
alpha_deg, CL_data, CD_data = data[:, 0], data[:, 1], data[:, 2]

def lift_drag_coeffs(alpha):
    a_deg = np.degrees(alpha)
    return np.interp(a_deg, alpha_deg, CL_data), np.interp(a_deg, alpha_deg, CD_data)
```

4. Re-run sections 5 to 7 for each airfoil and store the predicted glide distances below.

In [ ]:
# Fill this in as you test each airfoil
results = {
    # "NACA 0012": {"predicted_m": None, "measured_m": None},
    # "NACA 2412": {"predicted_m": None, "measured_m": None},
}
results

## 9. Predicted versus measured

After building and testing the real glider, record your measured distances above and compare.

**Why might they differ?** *manufacturing error, turbulence, surface finish, polars not matching real Reynolds number, launch inconsistency, centre of gravity...*

**What I would change next:** *...*

## Log

| Date | What I did | Prediction | Result | Next step |
|------|------------|------------|--------|-----------|
|      |            |            |        |           |